In [1]:
import nest_asyncio
nest_asyncio.apply()
print("✅ nest_asyncio 初始化成功，已允许单元格内嵌套运行 asyncio")

✅ nest_asyncio 初始化成功，已允许单元格内嵌套运行 asyncio


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

required_keys = ["TQ_USERNAME", "TQ_PASSWORD", "SIMNOW_INVESTOR_ID", "SIMNOW_PASSWORD", "SIMNOW_BROKER_ID"]
missing = [k for k in required_keys if not os.getenv(k)]

if missing:
    print(f"❌ 缺少环境变量: {missing}，请检查 .env 配置文件！")
else:
    print("✅ 所有核心交易与行情账号变量验证成功")

✅ 所有核心交易与行情账号变量验证成功


In [3]:
try:
    import openctp_ctp
    from tqsdk import TqApi
    print("✅ 依赖库 openctp-ctp 与 tqsdk 均已成功安装且可导入")
except ImportError as e:
    print(f"❌ 导入失败: {e}")

✅ 依赖库 openctp-ctp 与 tqsdk 均已成功安装且可导入


在使用天勤量化之前，默认您已经知晓并同意以下免责条款，如果不同意请立即停止使用：https://www.shinnytech.com/blog/disclaimer/


In [4]:
import os
import time
from datetime import datetime
from tqsdk import TqApi, TqAuth
from core.models import BarData
from strategy.strategies.double_ma_strategy import DoubleMaStrategy

# 连接天勤 API 并解析实际主力合约
tq_username = os.getenv("TQ_USERNAME")
tq_password = os.getenv("TQ_PASSWORD")
api = TqApi(auth=TqAuth(tq_username, tq_password))

quote = api.get_quote("KQ.m@SHFE.rb")
while not quote.underlying_symbol:
    api.wait_update()
main_contract = quote.underlying_symbol
print(f"🎯 主力合约映射: KQ.m@SHFE.rb -> {main_contract}")

2026-07-06 22:54:57 -     INFO - 通知 : 与 wss://free-api.shinnytech.com/t/nfmd/front/mobile 的网络连接已建立
🎯 主力合约映射: KQ.m@SHFE.rb -> SHFE.rb2610


In [5]:
import pandas as pd
print("📥 正在下载最近 1500 根 1m K线并时区对齐...")
klines = api.get_kline_serial("KQ.m@SHFE.rb", 60, data_length=1500)
api.wait_update(deadline=time.time() + 5)

# 校准时间戳与对齐
klines['db_datetime'] = pd.to_datetime(klines['datetime']) + pd.Timedelta(hours=8) + pd.Timedelta(minutes=1)

# 实例化策略
class DummyCtaEngine:
    def send_order(self, strategy, direction, offset, price, volume, stop=False):
        print(f"🛒 发送报单模拟: {direction.value} {offset.value} @ {price} x {volume}")
        return "mock_order_id"

engine = DummyCtaEngine()
strategy = DoubleMaStrategy(engine, "DoubleMa_Notebook_Test", "RB88.SHF", {"fast_window": 10, "slow_window": 20})

for _, row in klines.iterrows():
    bar = BarData(
        symbol="RB88",
        exchange="SHF",
        datetime=row['db_datetime'].to_pydatetime(),
        volume=row['volume'],
        open_price=row['open'],
        high_price=row['high'],
        low_price=row['low'],
        close_price=row['close']
    )
    strategy.on_bar(bar)
    
print(f"🔥 预热完成！当前策略快轨均线 MA0: {strategy.fast_ma0:.2f} | 慢轨均线 MA0: {strategy.slow_ma0:.2f}")
api.close()
print("🔌 天勤 API 连接已成功关闭，无泄露")

📥 正在下载最近 1500 根 1m K线并时区对齐...


🔥 预热完成！当前策略快轨均线 MA0: 3072.20 | 慢轨均线 MA0: 3071.10
🔌 天勤 API 连接已成功关闭，无泄露


In [6]:
import time
from core.event_engine import EventEngine
from gateway.ctp_gateway import CtpGateway

event_engine = EventEngine()
event_engine.start()

ctp_gateway = CtpGateway(event_engine)
print("🔌 正在发起 CTP 行情与交易双向连通登录...")
ctp_gateway.connect()

# 等待登录回调完成
time.sleep(3)
print(f"📶 行情连通状态: {ctp_gateway.md_logged_in} | 交易连通状态: {ctp_gateway.td_logged_in}")

if not ctp_gateway.md_logged_in or not ctp_gateway.td_logged_in:
    print("⚠️ [CTP网关] CTP 行情或交易登录失败，正在触发自动清理和资源释放！")
    print("🔍 常见排查步骤:")
    print("  1. 检查是否在交易时间内 (非交易时间SimNow前置机可能处于关闭/维护状态)。")
    print("  2. 检查项目根目录下 .env 文件中的 CTP 账户密码配置 (SIMNOW_INVESTOR_ID, SIMNOW_PASSWORD) 是否正确。")
    print("  3. 确认 SimNow 行情/交易前置地址 (SIMNOW_MD_FRONT, SIMNOW_TRADE_FRONT) 是否发生变更。")
    print("  4. 检查本地网络环境是否正常，是否存在防火墙或代理拦截端口。")
    print("🔌 正在自动释放 CTP 网关并停止事件引擎...")
    try:
        ctp_gateway.close()
    except Exception as e:
        print(f"释放网关异常: {e}")
    try:
        await event_engine.stop()
    except Exception as e:
        print(f"停止事件引擎异常: {e}")
    print("✅ 资源已安全清理完毕。")

🔌 正在发起 CTP 行情与交易双向连通登录...
🔌 [CTP网关] 正在建立行情前置连接: tcp://180.168.146.187:10131 ...
🔌 [CTP网关] 正在建立交易前置连接: tcp://180.168.146.187:10130 ...


📶 行情连通状态: False | 交易连通状态: False
⚠️ [CTP网关] CTP 行情或交易登录失败，正在触发自动清理和资源释放！
🔍 常见排查步骤:
  1. 检查是否在交易时间内 (非交易时间SimNow前置机可能处于关闭/维护状态)。
  2. 检查项目根目录下 .env 文件中的 CTP 账户密码配置 (SIMNOW_INVESTOR_ID, SIMNOW_PASSWORD) 是否正确。
  3. 确认 SimNow 行情/交易前置地址 (SIMNOW_MD_FRONT, SIMNOW_TRADE_FRONT) 是否发生变更。
  4. 检查本地网络环境是否正常，是否存在防火墙或代理拦截端口。
🔌 正在自动释放 CTP 网关并停止事件引擎...
🔌 [CTP网关] 行情接口已释放。
🔌 [CTP网关] 交易接口已释放。
✅ 资源已安全清理完毕。


In [7]:
# 订阅实际主力合约，并监控打印首批 Tick / 离线时进行模拟
import time
import asyncio
from datetime import datetime, timedelta
from core.event_engine import Event
from core.models import TickData

# 重启事件引擎以确保任务运行在当前单元格的 stdout 上下文中
if event_engine._active:
    await event_engine.stop()
event_engine._queue = asyncio.Queue()
event_engine.start()
print("✅ 事件引擎已在 Cell 7 重启并绑定当前 stdout 上下文")

# 获取合约基本信息
if 'main_contract' in globals() and main_contract:
    clean_code = main_contract.split('.')[1] if '.' in main_contract else main_contract
    exchange = main_contract.split('.')[0] if '.' in main_contract else "SHF"
else:
    clean_code = "rb2610"
    exchange = "SHF"

tick_list = []

def on_tick_event(event):
    tick = event.data
    if len(tick_list) < 5:
        tick_list.append(tick)
        if ctp_gateway.md_logged_in:
            print(f"📊 [CTP Tick] 价格: {tick.last_price} | 成交量增量: {tick.volume} | 买一: {tick.bid_price_1} | 时间: {tick.datetime}")
        else:
            print(f"📊 [模拟 CTP Tick] 价格: {tick.last_price} | 成交量增量: {tick.volume} | 买一: {tick.bid_price_1} | 时间: {tick.datetime}")

event_engine.register("eTick.", on_tick_event)

if ctp_gateway.md_logged_in:
    print(f"📥 正在为合约 {clean_code}.{exchange} 订阅 CTP 实时行情...")
    ctp_gateway.subscribe(clean_code, exchange)
    print("⏳ 正在等待接收前 5 笔实时 Tick 数据...")
    # 异步等待 5 秒观察是否有实时行情流入
    await asyncio.sleep(5)
else:
    print("⚠️ [CTP网关] 检测到 CTP 未登录 (SimNow 可能处于离线或维护状态)。")
    print("💡 将启用本地行情模拟以验证 Tick 订阅和事件分发逻辑...")
    
    # 模拟生成 5 个 Tick 并在几毫秒间隔内放入 event_engine
    base_time = datetime.now()
    for i in range(5):
        mock_tick = TickData(
            symbol=clean_code.upper(),
            exchange=exchange,
            datetime=base_time + timedelta(seconds=i),
            last_price=3000.0 + i * 2.0,
            volume=10.0 + i,
            turnover=30000.0 + i * 20.0,
            open_interest=10000.0,
            limit_up=3500.0,
            limit_down=2500.0,
            bid_price_1=2999.0 + i * 2.0,
            bid_volume_1=5.0,
            ask_price_1=3001.0 + i * 2.0,
            ask_volume_1=5.0
        )
        event_engine.put(Event("eTick.", mock_tick))
    
    # 异步等待事件处理完成，让事件循环有空档处理 _run 协程
    await asyncio.sleep(1)


✅ 事件引擎已在 Cell 7 重启并绑定当前 stdout 上下文
⚠️ [CTP网关] 检测到 CTP 未登录 (SimNow 可能处于离线或维护状态)。
💡 将启用本地行情模拟以验证 Tick 订阅和事件分发逻辑...
📊 [模拟 CTP Tick] 价格: 3000.0 | 成交量增量: 10.0 | 买一: 2999.0 | 时间: 2026-07-06 22:55:00.514791
📊 [模拟 CTP Tick] 价格: 3002.0 | 成交量增量: 11.0 | 买一: 3001.0 | 时间: 2026-07-06 22:55:01.514791
📊 [模拟 CTP Tick] 价格: 3004.0 | 成交量增量: 12.0 | 买一: 3003.0 | 时间: 2026-07-06 22:55:02.514791
📊 [模拟 CTP Tick] 价格: 3006.0 | 成交量增量: 13.0 | 买一: 3005.0 | 时间: 2026-07-06 22:55:03.514791
📊 [模拟 CTP Tick] 价格: 3008.0 | 成交量增量: 14.0 | 买一: 3007.0 | 时间: 2026-07-06 22:55:04.514791


In [8]:
# 接入 K 线生成器，模拟/接收 1m Bar 的生成与闭合
import time
import asyncio
from datetime import datetime, timedelta
from core.event_engine import Event
from core.models import TickData
from strategy.bar_generator import BarGenerator

def on_new_bar(bar):
    print(f"📦 [1m K线闭合] 均线更新触发源 -> 时间: {bar.datetime.strftime('%H:%M:%S')} | 收盘价: {bar.close_price} | 成交量: {bar.volume}")
    
bg = BarGenerator(on_bar=on_new_bar)

# 重启事件引擎以确保任务运行在当前单元格的 stdout 上下文中
if event_engine._active:
    await event_engine.stop()
event_engine._queue = asyncio.Queue()
event_engine.start()
print("✅ 事件引擎已在 Cell 8 重启并绑定当前 stdout 上下文")

event_engine.register("eTick.", lambda e: bg.update_tick(e.data))

# 获取合约基本信息
if 'main_contract' in globals() and main_contract:
    clean_code = main_contract.split('.')[1] if '.' in main_contract else main_contract
    exchange = main_contract.split('.')[0] if '.' in main_contract else "SHF"
else:
    clean_code = "rb2610"
    exchange = "SHF"

if ctp_gateway.md_logged_in:
    print("📡 聚合引擎已就绪，正在监听 Tick 转化为 Bar ... (等待 60 秒观察闭合，若无行情可先跳过)")
    await asyncio.sleep(60)
else:
    print("📡 [模拟] 聚合引擎已就绪，开始模拟发送跨越分钟的 Tick 数据以触发 Bar 闭合...")
    base_time = datetime.now().replace(second=0, microsecond=0)
    
    # 模拟发送 5 笔 tick，其中前 3 笔在第一分钟，后 2 笔在第二分钟以触发闭合
    # 第一分钟
    for i in range(3):
        mock_tick = TickData(
            symbol=clean_code.upper(),
            exchange=exchange,
            datetime=base_time + timedelta(seconds=i * 10),
            last_price=3000.0 + i,
            volume=5.0,
            turnover=15000.0,
            open_interest=10000.0,
            limit_up=3500.0,
            limit_down=2500.0,
            bid_price_1=2999.0 + i,
            bid_volume_1=5.0,
            ask_price_1=3001.0 + i,
            ask_volume_1=5.0
        )
        event_engine.put(Event("eTick.", mock_tick))
        await asyncio.sleep(0.1)
        
    # 第二分钟，分钟发生改变，触发闭合
    for i in range(2):
        mock_tick = TickData(
            symbol=clean_code.upper(),
            exchange=exchange,
            datetime=base_time + timedelta(minutes=1, seconds=i * 10),
            last_price=3010.0 + i,
            volume=5.0,
            turnover=15050.0,
            open_interest=10000.0,
            limit_up=3500.0,
            limit_down=2500.0,
            bid_price_1=3009.0 + i,
            bid_volume_1=5.0,
            ask_price_1=3011.0 + i,
            ask_volume_1=5.0
        )
        event_engine.put(Event("eTick.", mock_tick))
        await asyncio.sleep(0.1)
        
    await asyncio.sleep(1) # 等待事件引擎分发完成

# 运行清理，关闭网关和事件引擎，防止资源泄露
print("🔌 正在自动释放 CTP 网关并停止事件引擎...")
try:
    ctp_gateway.close()
except Exception as e:
    print(f"释放网关异常: {e}")
try:
    await event_engine.stop()
except Exception as e:
    print(f"停止事件引擎异常: {e}")
print("✅ 资源已安全清理完毕。")


✅ 事件引擎已在 Cell 8 重启并绑定当前 stdout 上下文
📡 [模拟] 聚合引擎已就绪，开始模拟发送跨越分钟的 Tick 数据以触发 Bar 闭合...


📦  K线闭合] 均线更新触发源 -> 时间: 22:55:00 | 收盘价: 3002.0 | 成交量: 15.0


🔌 正在自动释放 CTP 网关并停止事件引擎...
✅ 资源已安全清理完毕。


In [9]:
import os
import time
from tqsdk import TqApi, TqAuth
from gateway.ctp_gateway import CtpGateway
from core.event_engine import EventEngine

# Ensure api is re-initialized (since it was closed in Cell 5)
try:
    if 'api' in globals() and api:
        api.close()
except Exception:
    pass

tq_username = os.getenv("TQ_USERNAME")
tq_password = os.getenv("TQ_PASSWORD")
api = TqApi(auth=TqAuth(tq_username, tq_password))

# Ensure event_engine and ctp_gateway are defined
if 'event_engine' not in globals() or event_engine is None:
    event_engine = EventEngine()
    event_engine.start()
elif not event_engine._active:
    event_engine.start()

if 'ctp_gateway' not in globals() or ctp_gateway is None:
    ctp_gateway = CtpGateway(event_engine)

# 开启容灾切换检测变量
last_tick_time = time.time()
use_backup = False

# 订阅备用天勤行情数据
tq_klines = api.get_kline_serial("KQ.m@SHFE.rb", 60)

# 灾备自愈守护任务
def check_failover():
    global use_backup
    if time.time() - last_tick_time > 5: # 测试时设定 5 秒无 Tick 即切换
        if not use_backup:
            print("🚨 [灾备自愈] 检测到 CTP 网关已断开连接！正在切换至天勤备份源...")
            use_backup = True
            
# 运行并断开 CTP
print("🔌 主动切断 CTP 连接以模拟网络故障...")
ctp_gateway.close()
time.sleep(6)
check_failover()


2026-07-06 22:55:03 -     INFO - 通知 : 与 wss://free-api.shinnytech.com/t/nfmd/front/mobile 的网络连接已建立
🔌 主动切断 CTP 连接以模拟网络故障...


🚨 [灾备自愈] 检测到 CTP 网关已断开连接！正在切换至天勤备份源...


In [10]:
import os
import asyncio
from core.models import Direction, Offset
from core.trading_engine import TradingEngine

# 实例化真实主引擎，进入 Dry-Run 模式
live_main_engine = TradingEngine()
live_main_engine.dry_run = True
live_main_engine.start()

# 伪造一个委托请求
class MockOrderReq:
    symbol = "RB88"
    exchange = "SHF"
    direction = Direction.LONG
    offset = Offset.OPEN
    price = 3060.0
    volume = 2.0
    
print("🛒 触发策略金叉信号，投递买入请求...")
live_main_engine.send_order(MockOrderReq(), gateway_name="CTP")

# 验证日志生成
log_path = "logs/trading_signals.log"
if os.path.exists(log_path):
    print("\n📄 logs/trading_signals.log 最新日志内容：")
    with open(log_path, "r", encoding="utf-8") as f:
        print(f.readlines()[-1])

# 清理 API 资源
api.close()
await live_main_engine.stop()
await event_engine.stop()
print("🔌 所有测试资源已安全释放")


启动事件引擎...
交易系统主引擎启动完毕
🛒 触发策略金叉信号，投递买入请求...

⚠️  [DRY-RUN 信号拦截] 合约: RB88.SHF | 方向: 多 | 偏移: 开 | 价格: 3060.0 | 数量: 2.0


📄 logs/trading_signals.log 最新日志内容：
[2026-07-06 22:55:09] DRY-RUN SIGNAL - 合约: RB88.SHF | 方向: 多 | 价格: 3060.0 | 数量: 2.0

停止事件引擎...
交易系统主引擎已安全停止
🔌 所有测试资源已安全释放
